# SingleStore Projections: A Practical Guide

A **projection** is a copy of a columnstore table (possibly a column subset) that can be sharded and/or sorted differently from the base table. SingleStore keeps projections in sync with the base table automatically.

**Key facts:**
- Columnstore tables only (source and projection).
- `CREATE PROJECTION` is an **offline operation** — writes to the base table are blocked while it runs.
- Projections roughly double storage and update cost, so use them sparingly (typically ≤ a few per table).
- The query hint `use_projection='name'` is currently **required** for range filter optimization, and recommended any time you want to guarantee the optimizer picks the projection.

## 1. Setup

Create a sample columnstore table and populate it with data.

In [ ]:
%%sql
CREATE DATABASE IF NOT EXISTS demo;
USE demo;

In [ ]:
%%sql
DROP TABLE IF EXISTS orders;

CREATE TABLE orders (
    order_id    BIGINT NOT NULL,
    customer_id BIGINT NOT NULL,
    status      VARCHAR(50),
    region      VARCHAR(50),
    amount      DECIMAL(10, 2),
    created_at  DATETIME,
    SHARD KEY (order_id)
);

In [ ]:
%%sql
INSERT INTO orders (order_id, customer_id, status, region, amount, created_at)
VALUES
    (1, 101, 'shipped',   'west',  120.00, '2024-01-15 10:00:00'),
    (2, 102, 'pending',   'east',   45.50, '2024-02-20 11:30:00'),
    (3, 103, 'delivered', 'west',  300.00, '2024-03-05 09:15:00'),
    (4, 101, 'shipped',   'north', 200.00, '2024-03-10 14:00:00'),
    (5, 104, 'cancelled', 'south',  75.00, '2024-04-01 16:45:00'),
    (6, 102, 'delivered', 'east',  500.00, '2024-04-18 08:00:00'),
    (7, 103, 'pending',   'west',   90.00, '2024-05-22 13:20:00'),
    (8, 105, 'shipped',   'north', 150.00, '2024-06-30 17:00:00');

## 2. Create Projections

**Syntax:**
```sql
CREATE PROJECTION [IF NOT EXISTS] <projection_name>
    ([SHARD KEY(<cols>),] [SORT KEY(<cols>),] [KEY(<cols>),...])
AS SELECT <col_list>
FROM <table>;
```

> **Note:** `SELECT *` is allowed but the column list is fixed at creation time — new base table columns are **not** automatically added to the projection.

In [ ]:
%%sql
-- Projection 1: Optimise range filters / ORDER BY on customer_id.
-- Shard + sort on customer_id so lookups hit a single partition.
CREATE PROJECTION IF NOT EXISTS orders_by_customer
    (SHARD KEY(customer_id), SORT KEY(customer_id))
AS SELECT customer_id, order_id, created_at, amount, status
FROM orders;

In [ ]:
%%sql
-- Projection 2: Optimise range filters on created_at.
-- Sort key on created_at lets the engine skip segments outside the date range.
CREATE PROJECTION IF NOT EXISTS orders_by_date
    (SHARD KEY(order_id), SORT KEY(created_at))
AS SELECT order_id, customer_id, region, created_at, amount, status
FROM orders;

In [ ]:
%%sql
-- Projection 3: Optimise GROUP BY / aggregations on region.
-- Sharding on region keeps grouping data local, eliminating cross-node shuffles.
CREATE PROJECTION IF NOT EXISTS orders_by_region
    (SHARD KEY(region), SORT KEY(region))
AS SELECT region, order_id, amount, status, created_at
FROM orders;

## 3. Inspect Projections

In [ ]:
%%sql
SHOW PROJECTIONS ON orders;

In [ ]:
%%sql
-- Without hint: optimizer uses base table (broadcast scan)
EXPLAIN SELECT order_id, status, amount, created_at
FROM   orders
WHERE  customer_id = 102
ORDER BY created_at DESC;

In [ ]:
%%sql
-- With hint: optimizer targets the projection
EXPLAIN SELECT order_id, status, amount, created_at
FROM   orders WITH (use_projection='orders_by_customer')
WHERE  customer_id = 102
ORDER BY created_at DESC;

## 4. Query Examples Using the Projection Hint

The `use_projection` hint is **required** for range filter optimisation and is best practice whenever you want to guarantee the correct projection is selected.

```sql
FROM <table> WITH (use_projection='<projection_name>')
```

> In `JOIN` queries, place the hint after the projected table, e.g. `INNER JOIN orders WITH (use_projection='...')  ON ...`

In [ ]:
%%sql
-- Query A: Single-customer lookup (orders_by_customer)
-- Hits only the single partition that owns customer_id = 102.
SELECT order_id, status, amount, created_at
FROM   orders WITH (use_projection='orders_by_customer')
WHERE  customer_id = 102
ORDER BY created_at DESC;

In [ ]:
%%sql
-- Query B: Date-range filter (orders_by_date)
-- Sort key on created_at lets the engine skip segments outside the range.
SELECT order_id, customer_id, amount
FROM   orders WITH (use_projection='orders_by_date')
WHERE  created_at BETWEEN '2024-01-01' AND '2024-04-30'
ORDER BY created_at;

In [ ]:
%%sql
-- Query C: Regional aggregation (orders_by_region)
-- Sharding on region means all rows for a region are local — no cross-node shuffle.
SELECT region,
       COUNT(*)    AS order_count,
       SUM(amount) AS total_amount,
       AVG(amount) AS avg_amount
FROM   orders WITH (use_projection='orders_by_region')
GROUP BY region;

In [ ]:
%%sql
-- Tip: Run SHOW PROFILE after a query to confirm which projection was used.
SHOW PROFILE;

## 5. Disabling Projection Use (Debugging / Baseline Testing)

Force the optimizer to use the **base table** even when projections exist — useful for benchmarking or debugging.

In [ ]:
%%sql
-- Per-query: disable_projection_replacement forces base table use
SELECT order_id, status, amount
FROM   orders WITH (disable_projection_replacement = 1)
WHERE  customer_id = 101;

In [ ]:
%%sql
-- Session/global: turn off all projections except those with an explicit hint
-- SET consider_secondary_projection = OFF;
-- SET consider_secondary_projection = ON;  -- restore default

## 6. Drop Projections

Use `IF EXISTS` to make migration scripts idempotent — no error is thrown if the projection does not exist.

> ⚠️ **`IF EXISTS` requires SingleStore v9.1 or later.** On older versions, omit `IF EXISTS` and handle missing-projection errors in your migration tooling instead.

> ⚠️ All projections on a table must be dropped before the base table itself can be dropped.

In [ ]:
%%sql
-- Safe drop (v9.1+): no error if projection doesn't exist
DROP PROJECTION IF EXISTS orders_by_region   ON orders;
DROP PROJECTION IF EXISTS orders_by_date     ON orders;
DROP PROJECTION IF EXISTS orders_by_customer ON orders;

-- Traditional drop (all versions) — errors if projection is missing:
-- DROP PROJECTION orders_by_customer ON orders;

In [ ]:
%%sql
-- Verify all projections are gone
SHOW PROJECTIONS ON orders;

## 7. Cleanup

In [ ]:
%%sql
DROP TABLE IF EXISTS orders;
-- DROP DATABASE IF EXISTS demo;